# Module 11 — Dataclasses, Enums, and Value Semantics

## Exercise 11.4 — Four value objects that cannot hold an invalid value

The theme: validation happens ONCE, at construction. After that, every function
receiving one of these types can trust it completely -- no re-checking, no
defensive code, no "what if the email is empty" branch anywhere.
Run:  python ex04_value_objects.py

---

**How to work through this.** Each task below is its own cell. Run them one at a
time and read the output before moving on; that is the whole advantage of a
notebook over a script. Where a cell asks for a prediction, write it before you
run anything. Being wrong on purpose in a place where it costs nothing is how
the correct model gets built.

---

# The concepts behind this exercise

Read this before the tasks. Every idea the tasks below use is explained here, so
you should not need to leave this notebook.

The code cells in this part are demonstrations rather than exercises. Run them,
change a value, run them again. That is the whole point of having them here
instead of in a document.

## Concept 1. `@dataclass`

In [ ]:
from dataclasses import dataclass, field

@dataclass
class Point:
    x: float
    y: float
    label: str = ""

That generates `__init__`, `__repr__`, and `__eq__`. The annotations are not
decoration — the decorator reads `__annotations__` at class creation time to
find the fields, which is why a field without an annotation is silently ignored:

In [ ]:
@dataclass
class Broken:
    x: int
    y = 0            # NO annotation -> a class attribute, NOT a field
                     # It will not appear in __init__, __repr__ or __eq__.

### The options that matter

```text
@dataclass(
    frozen=True,      # immutable: __setattr__ raises; also generates __hash__
    slots=True,       # 3.10+: generate __slots__, saving ~40% memory
    order=True,       # generate __lt__, __le__, __gt__, __ge__ from field order
    kw_only=True,     # 3.10+: all fields keyword-only at the call site
    eq=True,          # default; set False to keep identity comparison
    repr=True,        # default
)
```


**`frozen=True` should be your default.** It gives you `__hash__` for free,
makes aliasing bugs (Module 02) unrepresentable, and makes the object safe to
share between threads (Module 21) and to use as a dict key or cache key.

**`slots=True` costs nothing** for a data-shaped class and saves real memory
(Module 08). The exception is anything needing `cached_property` or `weakref`.

**`kw_only=True` for anything with more than three fields.**
`Config(30, 3, True, False)` is unreadable; `Config(timeout=30, retries=3, ...)`
is not.

**`order=True` compares fields in declaration order**, as a tuple. If that is
not the ordering you want, write `__lt__` yourself — silently sorting by the
wrong field is worse than not sorting.

### `field()`

In [ ]:
@dataclass
class Config:
    name: str
    tags: list[str] = field(default_factory=list)      # NOT `= []`
    _cache: dict = field(default_factory=dict, repr=False, compare=False)
    created: datetime = field(default_factory=datetime.now)
    version: int = field(default=1, metadata={"docs": "schema version"})

`default_factory` runs **per instance**, which is the fix for Module 02's
mutable-default trap. `@dataclass` refuses a mutable default outright:

In [ ]:
@dataclass
class Bad:
    items: list = []      # ValueError: mutable default <class 'list'> ...

That is one of the nicest things about dataclasses: a whole bug category becomes
a startup error.

`compare=False` excludes a field from `__eq__` and ordering — right for caches,
timestamps, and derived values. `repr=False` keeps secrets out of logs
(Module 08).

### `__post_init__`

In [ ]:
@dataclass(frozen=True)
class Rectangle:
    width: float
    height: float
    area: float = field(init=False)        # computed, not passed in

    def __post_init__(self) -> None:
        if self.width <= 0 or self.height <= 0:
            raise ValueError(f"dimensions must be positive: {self}")
        object.__setattr__(self, "area", self.width * self.height)
        # object.__setattr__ because frozen blocks the normal assignment.
        # This is the documented way to set computed fields on a frozen
        # dataclass, and it is the one place you should use it.

Validation in `__post_init__` means **an invalid instance cannot exist**
(Module 08's rule), and every caller — the constructor, the deserializer, the
test fixture — gets it.

### Working with instances

In [ ]:
from dataclasses import replace, asdict, astuple, fields

p2 = replace(p, x=10)          # a NEW instance with one field changed
asdict(p)                       # recursive dict; follows nested dataclasses
astuple(p)                      # recursive tuple
[f.name for f in fields(p)]     # introspection

`replace()` is how you "modify" a frozen dataclass, and it is the pattern that
makes immutability practical.

Note that `asdict()` **deep-copies** everything, including nested dataclasses,
lists and dicts. That is usually what you want and occasionally an expensive
surprise.

---

## Concept 2. Choosing a record type

| | `dict` | `NamedTuple` | `TypedDict` | `dataclass` | Pydantic |
|---|---|---|---|---|---|
| Fixed fields | no | yes | yes | yes | yes |
| Type-checked | no | yes | yes | yes | yes |
| Runtime validation | no | no | no | manual | **automatic** |
| Mutable | yes | no | yes | optional | optional |
| Attribute access | `d["x"]` | `t.x` | `d["x"]` | `o.x` | `o.x` |
| Methods | no | yes | no | yes | yes |
| Iterable/unpackable | keys | yes | keys | no | no |
| Cost | zero | zero | zero (a dict at runtime) | small | a dependency |

**The decision:**

- **`dict`** — genuinely dynamic keys, or data passing straight through.
- **`TypedDict`** — you must stay a `dict` (JSON in and out, an existing API)
  but want static checking. It *is* a dict at runtime; there is no validation.
- **`NamedTuple`** — a small immutable record that benefits from tuple
  behaviour: unpacking, indexing, use as a dict key. Returning multiple values
  from a function is the classic case.
- **`dataclass`** — the default for a record you own.
- **Pydantic** — data crossing a **trust boundary**: HTTP bodies, config files,
  message queues. It validates and coerces at runtime, which is exactly what a
  boundary needs and what a dataclass does not do. Module 28.

The distinction worth internalising: **a dataclass's type hints are checked by
mypy, not by Python.** `Point(x="not a number")` runs happily. Pydantic checks
at runtime. Use dataclasses inside your program, Pydantic at its edges.

---

## Concept 3. `Enum`

In [ ]:
from enum import Enum, IntEnum, StrEnum, auto, Flag

class Status(Enum):
    PENDING = "pending"
    ACTIVE = "active"
    CLOSED = "closed"

Status.ACTIVE           # <Status.ACTIVE: 'active'>
Status.ACTIVE.value     # 'active'
Status("active")        # lookup BY VALUE -> Status.ACTIVE
Status["ACTIVE"]        # lookup by NAME
list(Status)            # iterable, in definition order

Enums replace magic strings and give you three things a string cannot: a typo is
a `ValueError` at the boundary rather than a silent no-match, the valid set is
discoverable and iterable, and a type checker can verify exhaustiveness in a
`match`.

In [ ]:
class Priority(IntEnum):        # comparable and usable as an int
    LOW = 1
    HIGH = 3

Priority.HIGH > Priority.LOW    # True
sorted(tasks, key=lambda t: t.priority)

class Colour(StrEnum):          # 3.11+: IS a str, so it JSON-serialises
    RED = "red"

json.dumps({"c": Colour.RED})   # works; a plain Enum raises

class Perm(Flag):               # combinable
    READ = auto()
    WRITE = auto()
    ALL = READ | WRITE

Perm.READ in (Perm.READ | Perm.WRITE)     # True

`IntEnum` and `StrEnum` exist for interoperability with code that expects a
plain int or str — serialization, database columns, HTTP headers. Prefer plain
`Enum` unless you need that, because the looseness that makes them convenient
also lets `Status.ACTIVE == "active"` be True, which defeats part of the point.

Enums with behaviour are fine and underused:

In [ ]:
class Status(Enum):
    PENDING = "pending"
    ACTIVE = "active"

    @property
    def is_terminal(self) -> bool:
        return self is Status.CLOSED

    @classmethod
    def from_legacy_code(cls, code: int) -> "Status":
        return {0: cls.PENDING, 1: cls.ACTIVE}[code]

---

## Concept 4. Value semantics

A **value object** is defined by its contents, not its identity. Two `Money`
objects holding $5 are interchangeable; two `BankAccount` objects with a $5
balance are not.

| | Value object | Entity |
|---|---|---|
| Identity | its contents | an ID that outlives changes |
| Equality | field by field | by ID |
| Mutability | immutable | usually mutable |
| Examples | `Money`, `Point`, `DateRange`, `Email` | `User`, `Order`, `Account` |
| Build with | `@dataclass(frozen=True)` | `@dataclass` with an id field |

**Prefer value objects.** The benefits compound:

- Aliasing bugs cannot happen (Module 02).
- Hashable, so usable as dict keys and cache keys.
- Thread-safe for free (Module 21) — no lock can be forgotten if there is
  nothing to protect.
- Trivially testable: construct, assert, done. No setup, no teardown.
- Easy to reason about: a value that cannot change cannot change *behind you*.

The objection is allocation cost. For records at ordinary scale it does not
matter; measure before you let it drive the design (Module 23).

### Making illegal states unrepresentable

In [ ]:
# weak: every consumer must re-check
@dataclass
class Order:
    status: str
    shipped_at: datetime | None = None

# strong: the type enforces it
@dataclass(frozen=True)
class Pending: ...

@dataclass(frozen=True)
class Shipped:
    shipped_at: datetime          # cannot be absent

Order = Pending | Shipped

In the second version, "shipped with no timestamp" cannot be constructed, so no
code needs to handle it and no test needs to cover it. Combined with `match`
(Module 04), the type checker verifies you handled every case.

This is the highest-leverage idea in Part 2: **push invariants into types, so
that the checking happens once at construction rather than everywhere else
forever.**

---

---

# Now the exercise

You have everything you need. Work top to bottom, and where a cell asks for a
prediction, write it before you run anything.

## The concepts this exercise uses

These are the numbered sections of [the module README](../README.md). If a task below stops making sense, the section named next to it is the one to re-read.

- Section 1: `@dataclass`
- Section 2: Choosing a record type
- Section 3: `Enum`
- Section 4: Value semantics
- Section 5: Copy semantics revisited

> The teaching for this module currently lives in the README rather than in this notebook. Read it alongside these cells.

## Setup

Run this first. It is the imports and any shared values the tasks below need.

In [ ]:
from __future__ import annotations

from dataclasses import dataclass
from datetime import date, timedelta
from decimal import Decimal


# TODO 1 -----------------------------------------------------------------------

---

## `Email`

A syntactically valid email address.

In [ ]:
@dataclass(frozen=True)
class Email:
    """A syntactically valid email address.

    - normalise: strip whitespace, lowercase the DOMAIN only (the local part is
      case-sensitive per RFC 5321, even though almost every provider ignores
      that -- note the discrepancy in a comment and pick a side)
    - reject: empty, no @, more than one @, empty local or domain part, a
      domain with no dot, whitespace anywhere inside
    - expose: .local, .domain, .is_disposable (against a small blocklist)
    - __str__ returns the address

    Then answer: full RFC 5322 validation by regex is famously about 6000
    characters long and still not exactly right. What is the correct amount of
    validation for an email address in a real system, and what actually proves
    an address is valid?
    """

---

## `Percentage`

A value between 0 and 100 inclusive.

In [ ]:
@dataclass(frozen=True, order=True)
class Percentage:
    """A value between 0 and 100 inclusive.

    - construct from a percentage (75) or from a fraction (0.75) via a
      classmethod. Make it impossible to confuse the two -- that confusion is a
      real and expensive bug class.
    - reject out-of-range and non-finite values
    - .fraction property
    - of(amount) applying the percentage to a Decimal
    - __str__ as "75%" or "12.5%" (no trailing zeros)
    - arithmetic: adding two Percentages -- does that make sense? Decide, and
      either implement it with a clamping rule or refuse it. Write down why.
    """

---

## `DateRange`

A half-open interval [start, end).

In [ ]:
@dataclass(frozen=True)
class DateRange:
    """A half-open interval [start, end).

    - reject end <= start (empty and inverted ranges are both errors here --
      decide whether an empty range should be legal and justify it)
    - .days, __len__, __contains__(date), __iter__ over the dates
    - overlaps(other), intersection(other) -> DateRange | None,
      union(other) -> DateRange (raise if they do not touch -- why?)
    - split_by_month() -> list[DateRange]

    Half-open is not arbitrary. Explain in a comment what
    [Jan 1, Feb 1) + [Feb 1, Mar 1) gives you that inclusive ranges do not, and
    connect it to Module 03's slicing convention.
    """

---

## `Money`

Exact currency, from Module 03 -- now as a proper value object.

In [ ]:
@dataclass(frozen=True, order=True)
class Money:
    """Exact currency, from Module 03 -- now as a proper value object.

    - integer minor units + currency code
    - reject float construction
    - arithmetic only within one currency
    - multiplication by a quantity, never by another Money
    - allocate(n) splitting exactly, with no lost minor units
    - ordering only within one currency (what should comparing USD to EUR do --
      raise, or return NotImplemented? These give different behaviour in
      sorted(). Try both and pick.)
    """

---

## `verify`

_verify_

In [ ]:
def verify() -> None:
    e = Email("  Ada@Example.COM ")
    assert str(e) == "Ada@example.com", str(e)
    assert e.domain == "example.com"
    assert e.local == "Ada"
    assert e == Email("Ada@example.com")
    assert {e: 1}[Email("Ada@example.com")] == 1
    for bad in ["", "no-at-sign", "a@@b.com", "@b.com", "a@", "a@b", "a b@c.com"]:
        try:
            Email(bad)
        except ValueError:
            pass
        else:
            raise AssertionError(f"Email({bad!r}) should have been rejected")

    p = Percentage(75)
    assert p.fraction == Decimal("0.75")
    assert str(p) == "75%"
    assert Percentage.from_fraction(0.125).__str__() == "12.5%"
    assert p.of(Decimal("200.00")) == Decimal("150.00")
    assert Percentage(10) < Percentage(20)
    for bad in [-1, 101]:
        try:
            Percentage(bad)
        except ValueError:
            pass
        else:
            raise AssertionError(f"Percentage({bad}) should have been rejected")

    r = DateRange(date(2026, 1, 1), date(2026, 2, 1))
    assert len(r) == 31
    assert date(2026, 1, 15) in r
    assert date(2026, 2, 1) not in r, "half-open: the end is excluded"
    assert len(list(r)) == 31
    r2 = DateRange(date(2026, 1, 20), date(2026, 3, 1))
    assert r.overlaps(r2)
    assert r.intersection(r2) == DateRange(date(2026, 1, 20), date(2026, 2, 1))
    assert len(DateRange(date(2026, 1, 1), date(2026, 4, 1)).split_by_month()) == 3
    try:
        DateRange(date(2026, 2, 1), date(2026, 1, 1))
    except ValueError:
        pass
    else:
        raise AssertionError("inverted range should be rejected")

    m = Money.parse("19.99")
    assert str(m) == "$19.99"
    assert m + Money.parse("0.01") == Money.parse("20.00")
    assert m * 3 == Money.parse("59.97")
    parts = Money.parse("10.00").allocate(3)
    assert [str(p) for p in parts] == ["$3.34", "$3.33", "$3.33"]
    assert sum(parts[1:], parts[0]) == Money.parse("10.00")
    try:
        Money.parse(19.99)      # type: ignore[arg-type]
    except TypeError:
        pass
    else:
        raise AssertionError("float construction must be rejected")

    print("all value object checks passed")

---

## Run it

This is what running the original file did. Everything above must have been run first.

In [ ]:
if __name__ == "__main__":
    verify()

---

## Before you move on

- [ ] Every cell above ran, in order, on a fresh kernel.
- [ ] You wrote a prediction before running, wherever one was asked for.
- [ ] You can say in one sentence what each task was actually testing.
- [ ] Anything that surprised you is written down in `PROGRESS.md`.

Compare against the worked answers in `../solutions/` only after your own
attempt runs.